# NeoMscope — YOLOv11-det training on Kaggle

Trains the onion cell division detector on Kaggle's free GPU tier (T4 16GB or P100 16GB).
Expected runtime: **30–60 minutes** for 100 epochs on ~127 images, well within the 30 hr/week quota.

## Prerequisites (do these BEFORE running this notebook)

1. Create a Kaggle account if you don't have one.
2. **Enable GPU**: Settings (right sidebar) → Accelerator → GPU T4 x2 or P100.
3. **Internet on**: Settings → Internet → On (needed for `pip install ultralytics`).
4. Upload your dataset as a Kaggle Dataset, OR have a public download URL ready.
   The dataset should follow the YOLO det layout exported by Roboflow:
   ```
   onioncell/
   ├── images/{train,val,test}/*.jpg
   ├── labels/{train,val,test}/*.txt
   └── data.yaml
   ```

## After running

Download the resulting `best.pt` from `runs/yolo11s-det-onioncell/weights/`.
Place it in `models/pt/best.pt` on your dev PC, then run `tools/export_onnx.py`.

## Cell 1 — Verify GPU

In [ ]:
!nvidia-smi

## Cell 2 — Install Ultralytics

In [ ]:
!pip install -q 'ultralytics>=8.3'
import ultralytics; ultralytics.checks()

## Cell 3 — Get the dataset

**Option A** — Kaggle Dataset (recommended): if you uploaded as a Kaggle Dataset, attach it via the right sidebar (Add data) and uncomment the `cp -r` line below.

**Option B** — GitHub Release: paste your release URL into `DATASET_URL`.

In [ ]:
import os
from pathlib import Path

DATA_ROOT = Path('/kaggle/working/datasets/onioncell')
DATA_ROOT.parent.mkdir(parents=True, exist_ok=True)

# Option A — Kaggle Dataset (replace 'your-username/onioncell-dataset')
# !cp -r /kaggle/input/onioncell-dataset/* {DATA_ROOT.parent}/

# Option B — GitHub Release
DATASET_URL = 'https://github.com/domafordarwin/NeoMscope/releases/download/dataset-v1/onioncell.zip'
if not DATA_ROOT.exists():
    !wget -q {DATASET_URL} -O /tmp/onioncell.zip
    !unzip -q /tmp/onioncell.zip -d {DATA_ROOT.parent}

# Sanity check
for split in ['train', 'val', 'test']:
    n_imgs = len(list((DATA_ROOT / 'images' / split).glob('*.jpg')))
    n_lbls = len(list((DATA_ROOT / 'labels' / split).glob('*.txt')))
    print(f'{split}: {n_imgs} images, {n_lbls} labels')

## Cell 4 — Write data.yaml

Mirrors `training/data.yaml` from the repo. Kept inline so this notebook is self-contained.

In [ ]:
import yaml

data_cfg = {
    'path': str(DATA_ROOT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'names': {0: 'Inter', 1: 'Pro', 2: 'Meta', 3: 'Ana', 4: 'Telo'},
}
data_yaml = Path('/kaggle/working/data.yaml')
data_yaml.write_text(yaml.dump(data_cfg))
print(data_yaml.read_text())

## Cell 5 — Train YOLOv11s-det

Hyperparameters mirror `docs/02-design/features/aihat-yolo-port.design.md` §4.3.
Domain-specific augmentation: `degrees=180, flipud=0.5` (cells are rotation-invariant).

In [ ]:
from ultralytics import YOLO

model = YOLO('yolo11s.pt')   # det pretrained
results = model.train(
    data=str(data_yaml),
    epochs=100,
    patience=20,
    batch=16,
    imgsz=640,
    device=0,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3,
    # Augmentation (cell domain)
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=180,    # cells rotation-invariant
    translate=0.1, scale=0.5,
    fliplr=0.5, flipud=0.5,
    mosaic=1.0, mixup=0.1, close_mosaic=10,
    # Output
    project='/kaggle/working/runs',
    name='yolo11s-det-onioncell',
    plots=True,
    save_period=10,   # checkpoint every 10 epochs (R-11 mitigation)
)

## Cell 6 — Evaluate on test set

In [ ]:
best_pt = Path('/kaggle/working/runs/yolo11s-det-onioncell/weights/best.pt')
model = YOLO(str(best_pt))
metrics = model.val(data=str(data_yaml), split='test', plots=True)
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
for cls_id, name in [(0, 'Inter'), (1, 'Pro'), (2, 'Meta'), (3, 'Ana'), (4, 'Telo')]:
    print(f'  {name:>6}: mAP50={metrics.box.maps[cls_id]:.4f}')

## Cell 7 — Download artifacts

After this cell, find `best.pt` in the **Output** sidebar (right side) and download it.
Move it to `models/pt/best.pt` on your dev PC.

In [ ]:
from IPython.display import FileLink
FileLink(str(best_pt))

## Backup — train YOLOv8s-det (R-09 contingency)

If the Hailo HEF compilation later fails on the YOLOv11-det model, run this cell to
produce a YOLOv8 backup. The Hailo Model Zoo has the most maturity on YOLOv8.

In [ ]:
# Uncomment to train backup
# model_v8 = YOLO('yolov8s.pt')
# results_v8 = model_v8.train(
#     data=str(data_yaml), epochs=100, patience=20, batch=16, imgsz=640, device=0,
#     project='/kaggle/working/runs', name='yolov8s-det-onioncell',
#     hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
#     degrees=180, fliplr=0.5, flipud=0.5,
# )